# Exploratory Data Analysis

The objective of this notebook is to explore customer behavior patterns
and identify factors potentially associated with customer churn.

In [ ]:
from src import load_customers

df = load_customers()

df.head()

---
## Overall Churn

Before moving to stratify the churn in different situations, we visualize the total churn preseented in the dataset.

In [ ]:
from src import run_sql_file

overall_churn = run_sql_file("overall_churn.sql")

overall_churn

The dataset presents a churn rate of approximately 26.54%, indicating that customer retention may represent a significant business challenge.

---
## Churn by Contract



In [ ]:
contract_churn = run_sql_file("churn_by_contract.sql")

contract_churn

At first glance, customers under longer-term contracts exhibit substantially lower churn rates, especially those under two-year contracts.

However, this relationship may partially reflect a temporal effect. Customers under longer contracts may simply have had fewer opportunities to churn, particularly during the contractual commitment period.

To investigate whether the relationship persists after partially controlling for customer lifetime, we analyze churn rates across different tenure groups.

In [ ]:
from src import churn_by_tenure_contract

churn_by_tenure_contract_result = churn_by_tenure_contract(df)

churn_by_tenure_contract_result

Across nearly all tenure groups, customers under longer-term contracts continue
to exhibit lower churn rates relative to month-to-month customers.

This suggests that the observed relationship is not explained solely by
contractual lock-in effects and may indicate stronger long-term customer
retention among customers under annual contracts.

Some particularly relevant observations include:

- 51.94% of customers under month-to-month contracts churn within the first
  11 months;
- No customer currently under a two-year contract churned before reaching
  24 months of tenure.

One important limitation is that the dataset does not contain a temporal
history of contract changes. Therefore, it is not possible to determine
whether customers currently under two-year contracts previously used the
service under shorter contract types before migrating to a longer-term plan.

---
## Churn by Tenure

The previous analysis suggested that customers with longer tenure tend to
present lower churn rates.

To better visualize this relationship, we analyze churn rates across different
tenure values.

In [ ]:
from src import plot_churn_by_tenure

churn_by_tenure = run_sql_file("churn_by_tenure.sql")

plot_churn_by_tenure(churn_by_tenure)

As expected, churn rates generally decrease as tenure increases.

It is important to note, however, that the dataset represents a snapshot of
customers at a single point in time rather than a longitudinal follow-up of
individual customers. Therefore, the results indicate that customers with
higher tenure tend to exhibit lower churn rates, but do not necessarily imply
that churn probability mechanically decreases for a given customer over time.

To provide additional context for the churn curve, we also inspect the
distribution of customers across tenure values.

In [ ]:
from src import plot_customers_by_tenure

plot_customers_by_tenure(df)

The distribution reveals a strong concentration of customers in the first
months of tenure, followed by a relatively distributed customer base across
intermediate and high tenure values.

A secondary concentration can also be observed among customers with very high
tenure, suggesting the presence of a substantial group of long-term retained
customers.

A particularly sharp decline in churn can be observed during the first months
of the customer lifecycle. To further investigate this pattern, we inspect
customers with tenure between 1 and 6 months.

In [ ]:
churn_by_tenure[(churn_by_tenure["tenure"] <= 6) & (churn_by_tenure["tenure"] != 0)]

The churn rate falls from 61.99% in the first month to 47.00% by the third
month, representing a decrease of 14.99 percentage points.

By the sixth month, the churn rate reaches 36.36%, corresponding to a total
decrease of 25.63 percentage points relative to the first month. This
represents a relative reduction of approximately 41.35% in churn rate between
the first and sixth months of tenure.

Overall, the results suggest that customers who remain active beyond the
initial months tend to become substantially more stable over time, indicating
that the early stages of the customer lifecycle may be especially important
for retention efforts.

---
## Monthly Charges

We now investigate how churn behaves across different monthly charge levels.

To begin, we inspect the distribution of customers by monthly charges.

In [ ]:
from src import plot_histogram

plot_histogram(df, "MonthlyCharges", bins=30, title="Customer Count by Monthly Charge", xlabel="Monthly Charge ($)", ylabel="Customers")

The distribution reveals a strong concentration of customers around lower
monthly charges, particularly near the $20 range, followed by a broader spread
across intermediate and high charge values.

This pattern suggests the existence of different customer profiles and service
combinations within the dataset.

In [ ]:
monthly_charge_churn = run_sql_file("monthly_charges_churn.sql")

monthly_charge_churn

In [ ]:
from src import plot_line

plot_line(df=monthly_charge_churn, x="monthly_charge_group", y="churn_rate", title="Churn by Monthly Charge", xlabel="Monthly Charge ($)", ylabel="Churn Rate (%)")


Churn rates generally increase as monthly charges rise, although the relationship is not perfectly monotonic.

Customers in lower charge ranges exhibit substantially lower churn rates,
whereas intermediate and high charge groups tend to present considerably
higher churn levels.

The highest churn rates are concentrated between the 65-79 and 95-109 ranges,
all above 35%.

The decline observed in the 110+ range should be interpreted cautiously, since
this group contains substantially fewer customers relative to the central
ranges of the distribution.

In [ ]:
from src import plot_box

plot_box(df=df, x="Churn", y="MonthlyCharges", title="Monthly Charges by Churn Status", xlabel="Churn", ylabel="Monthly Charges ($)")

Customers who churned tend to present higher monthly charges overall.

The median monthly charge among churned customers is noticeably higher than that of retained customers, suggesting that higher-priced customers may be more likely to discontinue the service.

In [ ]:
from src import plot_scatter

plot_scatter(df=df, x="tenure", y="MonthlyCharges", hue="Churn", title="Monthly Charges vs Tenure", xlabel="Tenure (Months)", ylabel="Monthly Charges ($)")

The relationship between tenure and monthly charges does not present a clear linear structure. However, churned customers appear to be more concentrated in the region combining low tenure and high monthly charges.

This pattern suggests that customers paying higher prices during the early stages of their lifecycle may be particularly vulnerable to churn.

---
## Internet Service

The previous analyses suggested that differences in monthly charges may
partially reflect differences in service composition.

To better understand this relationship, we now investigate churn behavior
across internet service categories.

In [ ]:
internet_service_churn = run_sql_file("churn_by_internet_service.sql")

internet_service_churn

In [ ]:
from src import plot_bar

plot_bar(
    df=internet_service_churn,
    x="InternetService",
    y="churn_rate",
    title="Churn Rate by Internet Service",
    xlabel="Internet Service",
    ylabel="Churn Rate (%)"
)

Fiber optic presents the highest churn rate (41.89%) relative to DSL and
customers without internet service, with the latter group exhibiting a
substantially lower churn rate of only 7.40%.

To better understand whether this relationship may be associated with pricing,
we inspect the distribution of monthly charges across internet service types.

In [ ]:
plot_box(df=df, x="InternetService", y="MonthlyCharges", title="Monthly Charges by Internet Service", xlabel="Internet Service", ylabel="Monthly Charges ($)")

Fiber optic customers tend to present substantially higher monthly charges
relative to the other internet service categories. This reinforces the previously observed association between higher monthly
charges and higher churn rates.

Additionally, customers without internet service exhibit a very narrow range
of monthly charges, indicating a highly standardized pricing structure. This pattern further supports the idea that the dataset contains distinct customer segments associated with different service bundles and pricing structures.

---
## Tech Support

The previous analyses suggested that customer retention may be associated not only with pricing, but also with the adoption of specific service categories.

We now investigate whether technical support services are associated with different churn behaviors.

In [ ]:
tech_support_churn = run_sql_file("churn_by_tech_support.sql")

tech_support_churn

In [ ]:
plot_bar(df=tech_support_churn, x="TechSupport", y="churn_rate", title="Churn Rate by Tech Support", xlabel="Tech Support", ylabel="Churn Rate (%)")

Customers without technical support exhibit substantially higher churn rates
relative to customers subscribed to the service.

In [ ]:
import pandas as pd

pd.crosstab(
    df["InternetService"],
    df["TechSupport"],
    (df['Churn'] == 'Yes'),
    aggfunc="mean"
)

Among Fiber optic customers, churn reaches approximately 49% for customers
without technical support, compared to roughly 23% among those subscribed to
the service.

This suggests that technical support adoption may be strongly associated with
customer retention, particularly among customers under premium internet
services.

---
## Online Security

We now investigate whether online security services are associated with
different churn behaviors.

In [ ]:
online_security_churn = run_sql_file("churn_by_online_security.sql")

online_security_churn

In [ ]:
plot_bar(df=online_security_churn, x="OnlineSecurity", y="churn_rate", title="Churn Rate by Online Security", xlabel="Online Security", ylabel="Churn Rate (%)")

In [ ]:
pd.crosstab(
    df["InternetService"],
    df["OnlineSecurity"],
    (df['Churn'] == 'Yes'),
    aggfunc="mean"
)

In [ ]:
pd.crosstab(
    df["TechSupport"],
    df["OnlineSecurity"],
    (df['Churn'] == 'Yes'),
    aggfunc="mean"
)

The relationship observed for online security services is highly similar to
the previous technical support analysis, with customers subscribed to
additional services consistently exhibiting lower churn rates.

The combination analysis between technical support and online security
reinforces this pattern even further. Customers subscribed to neither service
present the highest churn rates, whereas customers subscribed to both services
exhibit substantially lower churn levels.

This recurring behavior suggests that churn may be associated more broadly
with overall service adoption and customer engagement rather than with a
single specific service category.

---
## Online Backup

To further evaluate the previously observed pattern, we briefly investigate whether online backup services exhibit a similar relationship with churn.

In [ ]:
online_backup_churn = run_sql_file("churn_by_online_backup.sql")

online_backup_churn

In [ ]:
pd.crosstab(
    df["InternetService"],
    df["OnlineBackup"],
    (df['Churn'] == 'Yes'),
    aggfunc="mean"
)

In [ ]:
pd.crosstab(
    df["TechSupport"],
    df["OnlineBackup"],
    (df['Churn'] == 'Yes'),
    aggfunc="mean"
)

In [ ]:
pd.crosstab(
    df["OnlineSecurity"],
    df["OnlineBackup"],
    (df['Churn'] == 'Yes'),
    aggfunc="mean"
)

The same general pattern continues to emerge across additional service
categories. Among customers with internet service subscriptions, customers
subscribed to a larger number of supplementary services consistently exhibit
lower churn rates.

This suggests that the relationship is likely not restricted to a single
specific service type, but may instead reflect broader customer engagement
and service adoption behavior.

At the same time, these results introduce an apparent contradiction in the
analysis. Customers subscribed to more services are expected to present higher
monthly charges, while previous analyses showed that higher monthly charges
are generally associated with higher churn rates.

One possible explanation is that customers with broader service adoption may
also tend to present higher tenure, which was previously associated with lower
churn rates. Therefore, the relationship between pricing, service adoption,
tenure, and churn may involve interacting effects rather than isolated
variables alone.

This relationship will be investigated further in subsequent analyses.

---
## Phone Service and Multiple Lines

Previous analyses suggested the existence of a distinct low-cost customer
segment, especially among customers without internet service subscriptions.

To better understand the profile of these customers, we briefly investigate
phone-related services and their relationship with churn behavior.

In [ ]:
phone_service_churn = run_sql_file("churn_by_phone_service.sql")

phone_service_churn

In [ ]:
plot_bar(
    df=phone_service_churn,
    x="PhoneService",
    y="churn_rate",
    title="Churn Rate by Phone Service",
    xlabel="Phone Service",
    ylabel="Churn Rate (%)"
)

In [ ]:
multiple_lines_churn = run_sql_file("churn_by_multiple_lines.sql")

multiple_lines_churn

In [ ]:
plot_bar(
    df=multiple_lines_churn,
    x="MultipleLines",
    y="churn_rate",
    title="Churn Rate by Multiple Lines",
    xlabel="Multiple Lines",
    ylabel="Churn Rate (%)"
)

In [ ]:
pd.crosstab(
    df["InternetService"],
    df["PhoneService"]
)

Phone-related services appear substantially less associated with churn behavior than internet-related services.

All customers without internet service subscriptions are concentrated within the `PhoneService = Yes` category, indicating that the previously identified low-churn segment is primarily composed of customers subscribed only to phone services.

At the same time, phone service categories themselves do not strongly separate customers by churn behavior. Customers with and without phone service exhibit similar churn rates, while customers with multiple lines present only moderately higher churn levels.

Taken together, these results reinforce the idea that internet service adoption and related service bundles are substantially more associated with customer churn behavior than phone-related services alone.
